# Chapter 8: Wrapping Up
## *Introduction to Machine Learning with Python*  Müller & Guido

---
> **Ringkasan Bab:** Bab penutup ini membahas cara mendekati masalah ML secara menyeluruh di dunia nyata  mulai dari framing masalah, deployment ke production, testing sistem, hingga panduan kemana harus belajar lebih lanjut.

## 8.1 Mendekati Masalah Machine Learning

Ketika menghadapi masalah ML baru, ikuti alur sistematis berikut:

```
1. Definisikan masalah dengan jelas
        ↓
2. Kumpulkan & pahami data
        ↓
3. Bangun baseline sederhana
        ↓
4. Feature engineering & preprocessing
        ↓
5. Pilih & latih model
        ↓
6. Evaluasi & tuning
        ↓
7. Deploy ke production
        ↓
8. Monitor & iterasi
```

### Pertanyaan Kunci Sebelum Mulai:

| Pertanyaan | Mengapa Penting |
|---|---|
| Apa yang ingin diprediksi? | Menentukan supervised vs unsupervised |
| Bagaimana "sukses" diukur? | Menentukan metrik evaluasi yang tepat |
| Apakah data tersedia? | Menentukan feasibilitas proyek |
| Seberapa sering model perlu diupdate? | Menentukan arsitektur deployment |
| Apa konsekuensi kesalahan prediksi? | Menentukan prioritas precision vs recall |

In [ ]:
# Contoh: Checklist Proyek ML
checklist = {
    "1. Problem Framing": [
        " Apakah pertanyaan bisnis sudah jelas?",
        " Apakah ini supervised atau unsupervised?",
        " Apakah ini klasifikasi atau regresi?",
        " Metrik evaluasi apa yang relevan?",
    ],
    "2. Data": [
        " Apakah data cukup banyak?",
        " Apakah data representatif?",
        " Apakah ada missing values? Bagaimana menanganinya?",
        " Apakah ada data leakage?",
    ],
    "3. Modeling": [
        " Apakah ada baseline yang digunakan?",
        " Apakah sudah menggunakan cross-validation?",
        " Apakah hyperparameter sudah di-tune?",
        " Apakah model overfitting atau underfitting?",
    ],
    "4. Deployment": [
        " Bagaimana model akan dipanggil (API, batch, real-time)?",
        " Apakah ada monitoring untuk drift?",
        " Bagaimana model akan di-update?",
    ],
}

for section, items in checklist.items():
    print(f"\n{'='*45}")
    print(f" {section}")
    print(f"{'='*45}")
    for item in items:
        print(f"  {item}")

## 8.2 Humans in the Loop

**Manusia tetap penting** dalam sistem ML — terutama untuk:

- **Labeling data:** banyak algoritma supervised butuh data berlabel manusia
- **Edge cases:** model ML sering gagal pada kasus yang tidak biasa
- **Accountability:** keputusan penting (medis, hukum, keuangan) butuh oversight manusia
- **Feedback loop:** manusia bisa memberi koreksi untuk melatih ulang model

### Active Learning

**Active Learning** adalah strategi di mana model **meminta label** untuk sampel yang paling informatif (paling tidak yakin), sehingga mengurangi jumlah data berlabel yang dibutuhkan.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

# Ilustrasi konsep: model paling tidak yakin di dekat decision boundary
X, y = make_classification(n_samples=300, n_features=2, n_redundant=0,
                            n_clusters_per_class=1, random_state=42)
X_train, X_pool, y_train, y_pool = train_test_split(X, y, test_size=0.8, random_state=0)

# Train SVM awal dengan data berlabel sedikit
svm = SVC(kernel='rbf', probability=True)
svm.fit(X_train, y_train)

# Hitung uncertainty untuk setiap sampel unlabeled
proba = svm.predict_proba(X_pool)
uncertainty = 1 - np.max(proba, axis=1)  # semakin tinggi = semakin tidak yakin

# Sampel paling tidak yakin → kandidat untuk di-label manusia
most_uncertain_idx = np.argsort(uncertainty)[::-1][:10]

plt.figure(figsize=(9, 6))
scatter = plt.scatter(X_pool[:, 0], X_pool[:, 1],
                      c=uncertainty, cmap='RdYlGn_r', alpha=0.6, s=30)
plt.scatter(X_pool[most_uncertain_idx, 0], X_pool[most_uncertain_idx, 1],
            s=150, facecolors='none', edgecolors='blue', linewidths=2,
            label='10 Sampel Paling Tidak Yakin (Active Learning)')
plt.scatter(X_train[:, 0], X_train[:, 1], c='black', s=60,
            marker='*', label='Data Berlabel (Training Awal)')
plt.colorbar(scatter, label='Uncertainty')
plt.title('Active Learning: Pilih Sampel yang Paling Tidak Pasti untuk Di-label')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"Data berlabel awal  : {len(X_train)} sampel")
print(f"Unlabeled pool      : {len(X_pool)} sampel")
print(f"Active learning minta label untuk {len(most_uncertain_idx)} sampel terpilih")

## 8.3 Dari Prototype ke Production

Ada jurang besar antara notebook eksperimen dan sistem production yang handal.

### Tahapan Deployment:

| Tahap | Deskripsi |
|---|---|
| **Prototype** | Notebook Jupyter, data statis, evaluasi offline |
| **Staging** | Script Python, data pipeline, pengujian integrasi |
| **Production** | API endpoint, monitoring, logging, CI/CD |

### Cara Menyimpan dan Memuat Model:

In [ ]:
import joblib
import pickle
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# Latih model + pipeline
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, random_state=42
)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('rf',     RandomForestClassifier(n_estimators=100, random_state=42))
])
pipe.fit(X_train, y_train)
print(f"Akurasi test sebelum save: {pipe.score(X_test, y_test):.4f}")

# --- Simpan model dengan joblib (direkomendasikan untuk scikit-learn) ---
joblib.dump(pipe, 'model_production.pkl')
print("\n Model tersimpan ke: model_production.pkl")

# --- Load kembali ---
model_loaded = joblib.load('model_production.pkl')
print(f" Model berhasil dimuat")
print(f"Akurasi test setelah load: {model_loaded.score(X_test, y_test):.4f}")
print("\nHasil identik → model tersimpan dengan benar!")

In [ ]:
# Simulasi: Model sebagai fungsi prediksi yang siap di-deploy
def predict_cancer(features: list) -> dict:
    """
    Fungsi prediksi yang siap dipakai di API production.
    
    Args:
        features: list of 30 numerical features
    Returns:
        dict dengan prediksi dan probabilitas
    """
    import numpy as np
    X = np.array(features).reshape(1, -1)
    pred = model_loaded.predict(X)[0]
    proba = model_loaded.predict_proba(X)[0]
    return {
        'prediction'      : int(pred),
        'label'           : cancer.target_names[pred],
        'probability_malignant': round(float(proba[0]), 4),
        'probability_benign'   : round(float(proba[1]), 4),
    }

# Test fungsi prediksi
sample = cancer.data[0].tolist()  # ambil satu sampel
result = predict_cancer(sample)

print("=== Simulasi API Prediksi ===")
for key, val in result.items():
    print(f"  {key:<30}: {val}")

print(f"\n(Label aktual: {cancer.target_names[cancer.target[0]]})")

## 8.4 Testing Sistem Production

Sistem ML di production butuh beberapa lapisan pengujian:

### 8.4.1 Tipe-Tipe Testing

| Tipe | Deskripsi | Contoh |
|---|---|---|
| **Unit Test** | Uji tiap komponen secara terpisah | Test preprocessing, test prediksi |
| **Integration Test** | Uji alur end-to-end | Test pipeline lengkap |
| **A/B Testing** | Bandingkan model baru vs lama di production | Traffic split 50/50 |
| **Shadow Mode** | Jalankan model baru secara diam-diam, bandingkan | Log prediksi tanpa mempengaruhi user |
| **Data Drift Detection** | Deteksi perubahan distribusi data input | Statistik monitoring |
| **Concept Drift Detection** | Deteksi perubahan hubungan input-output | Akurasi menurun secara bertahap |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulasi: Deteksi model drift (degradasi performa seiring waktu)
np.random.seed(42)
n_weeks = 20

# Akurasi minggu pertama stabil, lalu mulai turun (concept drift)
baseline_acc = 0.95
noise = np.random.normal(0, 0.01, n_weeks)
drift = np.array([0] * 12 + [i * 0.015 for i in range(1, 9)])  # drift mulai minggu ke-12
weekly_accuracy = np.clip(baseline_acc + noise - drift, 0, 1)

alert_threshold = 0.88

plt.figure(figsize=(11, 5))
weeks = range(1, n_weeks + 1)
colors = ['red' if acc < alert_threshold else 'steelblue' for acc in weekly_accuracy]
plt.bar(weeks, weekly_accuracy, color=colors, alpha=0.8)
plt.axhline(y=baseline_acc, color='green', linestyle='--', linewidth=2, label=f'Baseline ({baseline_acc})')
plt.axhline(y=alert_threshold, color='red', linestyle='--', linewidth=2, label=f'Alert Threshold ({alert_threshold})')
plt.axvline(x=12.5, color='orange', linestyle=':', linewidth=2, label='Concept Drift Mulai')
plt.xlabel('Minggu')
plt.ylabel('Akurasi')
plt.title('Monitoring Performa Model di Production (Simulasi Concept Drift)')
plt.legend()
plt.ylim(0.8, 1.0)
plt.xticks(list(weeks))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

alert_weeks = [w for w, acc in zip(weeks, weekly_accuracy) if acc < alert_threshold]
print(f"  Alert diaktifkan pada minggu: {alert_weeks}")
print("→ Model perlu di-retrain dengan data terbaru!")

## 8.5 Membangun Estimator Sendiri

Anda bisa membuat **custom estimator** yang kompatibel dengan scikit-learn API (bisa dipakai di Pipeline, GridSearchCV, dll.) dengan mewarisi kelas `BaseEstimator` dan `TransformerMixin` atau `ClassifierMixin`.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
import numpy as np

# --- Contoh 1: Custom Transformer ---
class LogTransformer(BaseEstimator, TransformerMixin):
    """
    Custom transformer yang menerapkan log(1+x) pada semua fitur.
    Compatible dengan scikit-learn Pipeline.
    """
    def fit(self, X, y=None):
        # Validasi input
        X = check_array(X)
        self.n_features_in_ = X.shape[1]
        return self  # selalu return self

    def transform(self, X):
        check_is_fitted(self)
        X = check_array(X)
        return np.log1p(X)

# Test custom transformer
X_test_custom = np.array([[1, 10, 100], [2, 20, 200], [3, 30, 300]])

log_t = LogTransformer()
X_log = log_t.fit_transform(X_test_custom)

print("Input asli:")
print(X_test_custom)
print("\nSetelah LogTransformer:")
print(X_log.round(3))

In [ ]:
# --- Contoh 2: Custom Classifier ---
class MeanThresholdClassifier(BaseEstimator, ClassifierMixin):
    """
    Classifier sederhana: prediksi berdasarkan apakah rata-rata fitur
    lebih besar dari threshold yang dipelajari dari data training.
    """
    def fit(self, X, y):
        X, y = check_X_y(X, y)
        self.classes_ = np.unique(y)
        # Threshold = rata-rata dari semua nilai di training set
        self.threshold_ = np.mean(X)
        return self

    def predict(self, X):
        check_is_fitted(self)
        X = check_array(X)
        return (np.mean(X, axis=1) > self.threshold_).astype(int)

# Test dalam pipeline
from sklearn.pipeline import make_pipeline
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score

cancer = load_breast_cancer()
custom_pipe = make_pipeline(
    StandardScaler(),
    MeanThresholdClassifier()
)

scores = cross_val_score(custom_pipe, cancer.data, cancer.target, cv=5)
print(f"Custom Classifier CV Accuracy: {scores.mean():.3f} ± {scores.std():.3f}")
print("\n Custom estimator kompatibel dengan Pipeline dan cross_val_score!")

## 8.6 Kemana Harus Belajar Lebih Lanjut

### 8.6.1 Teori & Matematika

Pemahaman teori yang lebih dalam membantu ketika:
- Debugging mengapa model tidak bekerja
- Memilih algoritma yang tepat
- Mengimplementasikan algoritma baru

**Sumber belajar teori yang direkomendasikan:**

In [ ]:
resources = {
    "📚 Buku Teori ML": [
        "The Elements of Statistical Learning (Hastie, Tibshirani, Friedman) - FREE online",
        "Pattern Recognition and Machine Learning (Bishop)",
        "Machine Learning: A Probabilistic Perspective (Murphy)",
        "Deep Learning (Goodfellow, Bengio, Courville) - FREE online",
    ],
    "🎓 Kursus Online": [
        "fast.ai — Practical Deep Learning for Coders (gratis)",
        "Coursera: Machine Learning Specialization (Andrew Ng)",
        "CS229 Stanford — Machine Learning (YouTube gratis)",
        "CS231n Stanford — Convolutional Neural Networks (YouTube gratis)",
        "Kaggle Learn — Micro-courses ML gratis",
    ],
    "🔧 Library Lanjutan": [
        "TensorFlow / Keras — Deep Learning",
        "PyTorch — Deep Learning (research-friendly)",
        "XGBoost / LightGBM / CatBoost — Gradient Boosting canggih",
        "Hugging Face Transformers — NLP state-of-the-art",
        "statsmodels — Statistik & econometrics",
    ],
    "🏆 Praktek & Komunitas": [
        "Kaggle — Kompetisi ML dengan dataset nyata",
        "Papers with Code — Implementasi paper terbaru",
        "ArXiv.org — Preprint paper ML terbaru",
        "Towards Data Science (Medium) — Artikel praktis",
        "Reddit: r/MachineLearning, r/learnmachinelearning",
    ]
}

for section, items in resources.items():
    print(f"\n{section}")
    print("-" * 55)
    for item in items:
        print(f"  • {item}")

### 8.6.2 Framework ML Lain

Scikit-learn bukan satu-satunya pilihan. Berikut landscape lengkap ekosistem ML Python:

| Framework | Fokus | Kelebihan |
|---|---|---|
| **scikit-learn** | Classical ML | Konsisten, lengkap, terdokumentasi baik |
| **XGBoost** | Gradient Boosting | Sangat cepat, sering menang kompetisi |
| **LightGBM** | Gradient Boosting | Lebih cepat dari XGBoost, hemat memori |
| **CatBoost** | Gradient Boosting | Handle fitur kategorik otomatis |
| **TensorFlow/Keras** | Deep Learning | Production-ready, deployed luas |
| **PyTorch** | Deep Learning | Fleksibel, disukai peneliti |
| **Hugging Face** | NLP/LLM | State-of-the-art pre-trained models |
| **PySpark MLlib** | Distributed ML | Big data (cluster computing) |
| **H2O.ai** | AutoML | Otomatisasi proses ML |

### 8.6.3 Ranking, Recommender Systems, dan Tipe ML Lain

Selain supervised dan unsupervised learning, ada tipe ML lain yang penting:

| Tipe | Deskripsi | Contoh Aplikasi |
|---|---|---|
| **Learning to Rank** | Optimasi urutan/ranking | Search engine, rekomendasi |
| **Collaborative Filtering** | Rekomendasi berbasis perilaku pengguna | Netflix, Spotify, Tokopedia |
| **Reinforcement Learning** | Belajar dari interaksi & reward | Game AI, robot, trading |
| **Self-Supervised Learning** | Label otomatis dari data itu sendiri | GPT, BERT pre-training |
| **Few-Shot Learning** | Belajar dari sangat sedikit contoh | LLM prompting |
| **Online Learning** | Update model secara real-time | Stream data, ads CTR |

In [ ]:
# Contoh sederhana: Collaborative Filtering (User-Based)
import numpy as np
import pandas as pd

# Matrix rating pengguna-film (0 = belum ditonton)
ratings = pd.DataFrame({
    'Avengers': [5, 4, 0, 1, 0],
    'Titanic' : [1, 0, 5, 4, 3],
    'Inception': [4, 5, 2, 0, 4],
    'Toy Story': [0, 2, 4, 5, 3],
    'Matrix'  : [5, 4, 1, 0, 5],
}, index=['Alice', 'Bob', 'Carol', 'David', 'Eve'])

print("Rating Matrix Pengguna-Film:")
print(ratings)

# Hitung cosine similarity antar pengguna
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(ratings)
sim_df = pd.DataFrame(sim_matrix, index=ratings.index, columns=ratings.index)

print("\nSimilaritas antar pengguna (Cosine Similarity):")
print(sim_df.round(2))

# Rekomendasikan film untuk Alice
alice_ratings = ratings.loc['Alice']
unseen = alice_ratings[alice_ratings == 0].index.tolist()

print(f"\nFilm yang belum ditonton Alice: {unseen}")

# Pengguna paling mirip dengan Alice (kecuali Alice sendiri)
alice_sim = sim_df['Alice'].drop('Alice').sort_values(ascending=False)
print(f"Pengguna paling mirip dengan Alice: {alice_sim.index[0]} (sim={alice_sim.iloc[0]:.2f})")

### 8.6.4 Probabilistic Modeling & Neural Networks Lanjutan

**Probabilistic Modeling** memungkinkan model untuk mengekspresikan **ketidakpastian** dalam prediksi sangat penting dalam domain kritis.

| Pendekatan | Tools | Kegunaan |
|---|---|---|
| **Bayesian ML** | PyMC, Stan | Uncertainty quantification |
| **Gaussian Processes** | GPy, scikit-learn | Regresi dengan uncertainty |
| **Variational AutoEncoder** | PyTorch/TF | Generative modeling |

**Neural Networks Lanjutan:**

| Arsitektur | Singkatan | Kegunaan Utama |
|---|---|---|
| Convolutional NN | CNN | Computer Vision (gambar) |
| Recurrent NN | RNN/LSTM | Sequence data, time series |
| Transformer | - | NLP, Vision, generasi teks |
| Generative Adversarial | GAN | Generasi gambar/teks |
| Large Language Model | LLM | GPT, BERT, Claude |
| Diffusion Model | - | Generasi gambar (Stable Diffusion) |

### 8.6.5 Scaling ke Dataset Besar

Ketika data sudah terlalu besar untuk masuk ke memori RAM:

In [ ]:
# Strategi scaling untuk dataset besar
scaling_strategies = {
    "1. Out-of-core learning (scikit-learn)": [
        "Gunakan model dengan .partial_fit() — latih secara batch/mini-batch",
        "Contoh: SGDClassifier, MiniBatchKMeans, PassiveAggressiveClassifier",
        "Data dibaca per batch, tidak perlu semua di RAM",
    ],
    "2. Dimensionality Reduction dulu": [
        "Gunakan PCA atau hashing untuk kompres fitur",
        "HashingVectorizer untuk teks (tidak butuh fit dulu)",
    ],
    "3. Distributed Computing": [
        "Apache Spark + MLlib — cluster multi-node",
        "Dask-ML — distributed scikit-learn",
        "Ray + Ray Tune — distributed hyperparameter search",
    ],
    "4. GPU Acceleration": [
        "RAPIDS cuML — scikit-learn API tapi di GPU",
        "PyTorch / TensorFlow — native GPU support",
        "XGBoost / LightGBM — bisa pakai GPU",
    ],
}

for strategy, tips in scaling_strategies.items():
    print(f"\n{strategy}")
    for tip in tips:
        print(f"  → {tip}")

In [ ]:
# Contoh: Out-of-core learning dengan SGDClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification
import numpy as np

# Simulasi: data besar yang datang dalam batch
np.random.seed(42)
TOTAL_SAMPLES = 10000
BATCH_SIZE    = 500

X_all, y_all = make_classification(n_samples=TOTAL_SAMPLES, n_features=20, random_state=42)

# SGD classifier mendukung partial_fit
sgd = SGDClassifier(loss='log_loss', random_state=42)

n_batches = TOTAL_SAMPLES // BATCH_SIZE
batch_accuracies = []

# Simulasi streaming data batch per batch
for i in range(n_batches):
    start = i * BATCH_SIZE
    end   = start + BATCH_SIZE
    X_batch = X_all[start:end]
    y_batch = y_all[start:end]
    
    # partial_fit: update model tanpa melatih ulang dari awal
    sgd.partial_fit(X_batch, y_batch, classes=[0, 1])
    
    # Evaluasi pada batch terkini
    acc = sgd.score(X_batch, y_batch)
    batch_accuracies.append(acc)

plt.figure(figsize=(10, 4))
plt.plot(range(1, n_batches+1), batch_accuracies, 'o-', color='steelblue', ms=4)
plt.xlabel('Batch')
plt.ylabel('Akurasi')
plt.title(f'Out-of-Core Learning: SGDClassifier ({n_batches} batch × {BATCH_SIZE} sampel)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Akurasi akhir: {batch_accuracies[-1]:.3f}")

## 8.7 Mengasah Kemampuan

Tips praktis untuk terus berkembang sebagai ML practitioner:

In [ ]:
tips_belajar = [
    {
        "level": "🌱 Pemula",
        "tips": [
            "Ikuti tutorial step-by-step di Kaggle Learn",
            "Reproduksi notebook dari proyek ML yang sudah ada",
            "Coba setiap algoritma pada dataset benchmark (Iris, MNIST, Titanic)",
            "Dokumentasikan setiap eksperimen di notebook dengan rapi",
        ]
    },
    {
        "level": "🌿 Menengah",
        "tips": [
            "Ikuti kompetisi Kaggle — bahkan tanpa menang, belajarnya luar biasa",
            "Baca solusi top dari kompetisi yang sudah selesai",
            "Bangun proyek end-to-end dari data mentah sampai deployment",
            "Mulai baca paper di ArXiv — mulai dari survey paper",
        ]
    },
    {
        "level": "🌳 Lanjut",
        "tips": [
            "Implementasikan algoritma dari paper secara manual",
            "Kontribusi ke open source (scikit-learn, PyTorch, dsb.)",
            "Tulis blog post — mengajar adalah cara belajar terbaik",
            "Spesialisasi di satu domain: CV, NLP, RL, atau time series",
        ]
    },
]

for level_info in tips_belajar:
    print(f"\n{level_info['level']}")
    print("-" * 50)
    for tip in level_info['tips']:
        print(f"  • {tip}")

## 8.8 Kesimpulan

Selamat telah menyelesaikan seluruh buku **Introduction to Machine Learning with Python**! 🎉

In [ ]:
# Recap perjalanan belajar kita
ringkasan_buku = {
    "Bab 1: Introduction": [
        "Library dasar: NumPy, Pandas, Matplotlib, SciPy, scikit-learn",
        "Konsep ML: supervised vs unsupervised",
        "First model: k-NN pada Iris dataset",
    ],
    "Bab 2: Supervised Learning": [
        "Klasifikasi & Regresi",
        "Linear models, Decision Trees, Random Forest, Gradient Boosting, SVM, MLP",
        "Overfitting, underfitting, regularisasi",
    ],
    "Bab 3: Unsupervised Learning": [
        "Preprocessing & Feature Scaling",
        "Dimensionality Reduction: PCA, NMF, t-SNE",
        "Clustering: K-Means, Agglomerative, DBSCAN",
    ],
    "Bab 4: Representing Data": [
        "One-Hot Encoding, Ordinal Encoding",
        "Binning, Polynomial Features",
        "Feature Selection & Imputation",
    ],
    "Bab 5: Model Evaluation": [
        "Cross-Validation (K-Fold, Stratified)",
        "GridSearchCV, RandomizedSearchCV",
        "Precision, Recall, F1, ROC-AUC",
    ],
    "Bab 6: Pipelines": [
        "Data Leakage & cara mencegahnya",
        "Pipeline & make_pipeline",
        "ColumnTransformer untuk data campuran",
    ],
    "Bab 7: Text Data": [
        "Bag of Words, TF-IDF",
        "N-Gram, Analisis Sentimen",
        "Topic Modeling (LDA)",
    ],
    "Bab 8: Wrapping Up": [
        "Pendekatan masalah ML secara sistematis",
        "Deployment & Testing production",
        "Roadmap belajar lanjutan",
    ],
}

print("=" * 58)
print("  RINGKASAN BUKU: Introduction to Machine Learning with Python")
print("=" * 58)

for bab, poin in ringkasan_buku.items():
    print(f"\n📘 {bab}")
    for p in poin:
        print(f"   ✓ {p}")

print("\n" + "=" * 58)
print("  Selamat! Anda kini siap mengerjakan proyek ML nyata! 🚀")
print("=" * 58)

##  Ringkasan Bab 8

| Konsep | Penjelasan |
|---|---|
| **ML Problem Framing** | Definisikan masalah, data, metrik sebelum coding |
| **Humans in the Loop** | Manusia tetap penting — labeling, oversight, feedback |
| **Active Learning** | Minta label pada sampel paling informatif |
| **Deployment** | Simpan model dengan `joblib`, bungkus dalam fungsi/API |
| **Production Testing** | Unit test, A/B test, monitoring drift |
| **Custom Estimator** | Buat transformer/classifier sendiri yang kompatibel sklearn |
| **Scaling** | `partial_fit`, Spark, Dask, GPU untuk data besar |
| **Next Steps** | Deep Learning, NLP lanjutan, Reinforcement Learning |

---